# Forecasting Final Project — Extended Fundamentals Analysis
## S&P 500 Energy Sector Index (S5ENRS) — Candidate Additional Fundamentals
### Econ 409 | University of California, Los Angeles
Jacob Williams · Praus Paek · Joshua Kenworthy · Agnibha Bhattacharya · Ignacio Ramirez

---

## 1. Introduction

The S&P 500 Energy Sector Index (S5ENRS) tracks U.S. large-cap energy companies and exhibits valuation cycles that often persist for several years, driven by commodity price dynamics, interest rate conditions, and broad market sentiment. This project constructs and evaluates two systematic weekly trading strategies for S5ENRS using a fundamentals-based fair value gap framework.

Rather than relying on price momentum or technical indicators, both strategies model fair value as a function of observable inputs — the U.S. 1-year Treasury yield, the S&P 500 forward P/E ratio, and the S&P 500 trailing P/E ratio — and generate signals when the observed price deviates substantially from that modelled fair value. The economic premise is that persistent deviations from fundamentals-implied fair value tend to partially mean-revert over subsequent periods.

Two estimation approaches are compared. The first uses a rolling OLS regression with a fixed trailing window, allowing the fair value relationship to adapt to recent data. The second uses a Kalman filter with time-varying coefficients, allowing the relationship to evolve continuously over time. Both strategies allow long, flat, and short weekly positions.

Performance is evaluated out-of-sample over approximately eight years (January 2018 – March 2026). The S&P 500 Index serves as the broad market benchmark; the S5ENRS buy-and-hold provides the sector-specific reference. Hyperparameters were selected using the training sample only; the test period was not used at any point during model development.

**Extension.** This notebook extends the original analysis by evaluating two candidate additional fundamentals: WTI crude oil price and the ICE BofA US High Yield Option-Adjusted Spread. WTI crude oil price is a direct revenue driver for energy sector firms, whose profitability is tightly linked to commodity prices. The high-yield credit spread captures macro-financial stress conditions that are particularly relevant to leveraged energy companies. Both series are evaluated as candidates using the same training-only variable selection framework; they are not assumed a priori to improve strategy performance. All hyperparameter selection, including feature set selection, is conducted on training data only.

---

## 2. Strategy and Conceptual Framework

### Fundamental Inputs

Three original observable variables serve as proxies for the fundamental value of the S5ENRS Index, with two new candidate variables added in this extension.

**U.S. 1-year Treasury yield (`t1y`).** Short-term rates affect equity valuations through the discount rate channel. Higher rates compress P/E multiples by raising the hurdle rate for equity returns. The energy sector is sensitive to this mechanism through both financing costs and broader economic demand conditions.

**S&P 500 forward P/E (`sp500_best_pe`).** The consensus forward P/E reflects market-wide earnings expectations. When broad market optimism is elevated, energy sector valuations tend to follow, creating scope for mean-reversion when the sector diverges from the broader trend.

**S&P 500 trailing P/E (`sp500_pe`).** The trailing P/E captures the relationship between realised earnings and current prices. Together with the forward P/E, it characterises the gap between market expectations and actual fundamentals.

The S&P 500 price level (`sp500_price`) is used only as a performance benchmark and does not enter any signal computation.

### New Candidate Fundamentals

**WTI crude oil price (`wti`).** The log of the WTI crude oil spot price (FRED: DCOILWTICO) is included as a candidate predictor. Energy sector revenues and earnings are directly tied to commodity prices, and sustained deviations of S5ENRS from oil-price-implied levels represent a plausible mean-reversion opportunity.

**High-yield credit spread (`hy_spread`).** The ICE BofA US High Yield Option-Adjusted Spread (FRED: BAMLH0A0HYM2) is a measure of macro-financial stress and risk appetite. Many energy companies carry significant leverage; periods of spread widening tend to compress energy sector valuations beyond what commodity prices alone would imply. This series is available from December 31, 1996 onward, which slightly shortens the usable overlap window relative to the original analysis.

Both new variables are evaluated as candidates in the training-period variable selection step. Their inclusion is not assumed beneficial; the selection procedure may return the baseline specification if the new inputs do not improve training performance.

### Strategy 1: Rolling Regression Fair Value Gap

At each week $t$, a rolling OLS regression of $\log(\text{S5ENRS}_t)$ on the selected inputs is estimated using the preceding $W$ observations. The fitted value gives the model's estimate of log fair value, constructed using only data through $t-1$. The fair value gap is the signed difference between observed and fitted log price, standardised by an expanding-window z-score. A long position is entered when the gap is sufficiently negative (price below fair value), a short position when the gap is sufficiently positive, and a flat position otherwise. The rolling window allows the fair value relationship to adapt to changing regimes.

### Strategy 2: Kalman Filter Fair Value Gap

The Kalman filter re-estimates the same regression in a state-space framework where the coefficient vector evolves as a random walk. At each $t$, the fair value estimate is the one-step-ahead prediction — the projection of $\log(\text{S5ENRS}_t)$ formed before incorporating the current-period observation. This is leakage-safe by construction. The same z-score threshold rule converts the gap into a directional signal.

### Position Space

Both strategies allow three positions: long ($+1$), flat ($0$), and short ($-1$). The flat position is taken when the standardised gap falls within the threshold band. This avoids forcing continuous exposure when the model does not produce a sufficiently clear signal, which is economically relevant for an energy sector strategy where the fundamental relationship may be temporarily unstable.

---

## 3. Leakage Prevention and Backtest Rules

The following constraints are enforced throughout all data preparation, signal construction, and backtesting steps. Violations of any of these rules would invalidate the backtest.

1. **Information timing.** All signals are formed exclusively from information available at time $t$. No observation from period $t+1$ or later may enter the signal computation for period $t$.

2. **Trade execution lag.** Signals formed using information through week $t$ are applied to the return realized over week $t+1$. No same-period return is used in strategy evaluation.

3. **No full-sample normalization.** Standardization, scaling, or normalization of any input series is performed only on data available up to and including $t$, using expanding or rolling statistics computed strictly in-sample. Full-sample statistics (e.g., z-scoring on the entire dataset) are prohibited.

4. **No test-set hyperparameter tuning.** All model hyperparameters (e.g., Kalman filter noise ratios, regression window lengths, signal thresholds, feature set selection) are selected using the training set only. The test set is not touched during model selection.

5. **No future observations in rolling or expanding calculations.** All rolling windows, expanding windows, and recursive estimators are configured with `min_periods` and applied strictly forward in time. No `.shift(-k)` for $k > 0$ may appear on the right-hand side of any feature computation.

6. **Benchmark separation.** The S&P 500 price series (`sp500_price`) is used only for performance benchmarking after strategy signals are finalized. It does not enter any forecasting model as an input or feature.

---

## 4. Methodology

### 4.1 Data and Variable Definitions

Five daily time series are loaded from local CSV files exported from Bloomberg, and two additional series are downloaded from FRED via `pandas_datareader`. Each Bloomberg file uses a distinct date format, so date parsing is handled individually per file.

| Variable | Source file | Role |
|---|---|---|
| `enrs` | S5ENGS.csv | Tradable asset: S&P 500 Energy Sector Index daily last price |
| `t1y` | H15T1Y.csv | Forecasting input: U.S. 1-year Treasury yield (%) |
| `sp500_best_pe` | SP500 Best PE.csv | Forecasting input: S&P 500 forward (consensus) P/E ratio |
| `sp500_pe` | SP500 PE.csv | Forecasting input: S&P 500 trailing P/E ratio |
| `sp500_price` | SP500 Price.csv | Benchmark only: S&P 500 Index daily last price |
| `wti` | wti_crude.csv | Candidate input: log WTI crude oil spot price (FRED: DCOILWTICO) |
| `hy_spread` | high_yield_credit_spread.csv | Candidate input: ICE BofA US HY OAS, % (FRED: BAMLH0A0HYM2) |

**Date format note.** `S5ENGS.csv` uses `M/D/YYYY`. All four SP500 and US Treasury files use `DD/MM/YYYY`. The FRED-sourced CSVs use ISO format (`YYYY-MM-DD`). Explicit `format` strings are passed to `pd.to_datetime` for the Bloomberg files. The FRED files use `pd.to_datetime` with default parsing.

**HY spread availability.** The `hy_spread` series begins on December 31, 1996. This restricts the usable overlap window for the extended analysis to approximately January 1997 onward, compared with approximately January 1990 in the original analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
enrs = pd.read_csv('S5ENGS.csv')
enrs.columns = enrs.columns.str.strip()
enrs = enrs.rename(columns={'S5ENRS Index - Last Price': 'enrs'})
enrs['date'] = pd.to_datetime(enrs['Date'], format='%m/%d/%Y', errors='coerce')
enrs['enrs'] = pd.to_numeric(enrs['enrs'], errors='coerce')
enrs = enrs[['date', 'enrs']].dropna().sort_values('date').reset_index(drop=True)
display(enrs.head(3))

In [ ]:
t1y = pd.read_csv('SP 500 and US Treasury/H15T1Y.csv')
t1y.columns = t1y.columns.str.strip()
t1y = t1y.rename(columns={'Last Price': 't1y'})
t1y['date'] = pd.to_datetime(t1y['Date'], format='%d/%m/%Y', errors='coerce')
t1y['t1y'] = pd.to_numeric(t1y['t1y'], errors='coerce')
t1y = t1y[['date', 't1y']].dropna().sort_values('date').reset_index(drop=True)
display(t1y.head(3))

In [ ]:
bpe = pd.read_csv('SP 500 and US Treasury/SP500 Best PE.csv')
bpe.columns = bpe.columns.str.strip()
bpe = bpe.rename(columns={'BEst P/E Ratio': 'sp500_best_pe'})
bpe['date'] = pd.to_datetime(bpe['Date'], format='%d/%m/%Y', errors='coerce')
bpe['sp500_best_pe'] = pd.to_numeric(bpe['sp500_best_pe'], errors='coerce')
bpe = bpe[['date', 'sp500_best_pe']].dropna().sort_values('date').reset_index(drop=True)
display(bpe.head(3))

In [ ]:
pe = pd.read_csv('SP 500 and US Treasury/SP500 PE.csv')
pe.columns = pe.columns.str.strip()
pe = pe.rename(columns={'Price Earnings Ratio (P/E)': 'sp500_pe'})
pe['date'] = pd.to_datetime(pe['Date'], format='%d/%m/%Y', errors='coerce')
pe['sp500_pe'] = pd.to_numeric(pe['sp500_pe'], errors='coerce')
pe = pe[['date', 'sp500_pe']].dropna().sort_values('date').reset_index(drop=True)
display(pe.head(3))

In [ ]:
sp500 = pd.read_csv('SP 500 and US Treasury/SP500 Price.csv')
sp500 = sp500.dropna(axis=1, how='all')
sp500.columns = sp500.columns.str.strip()
sp500 = sp500.rename(columns={'Last Price': 'sp500_price'})
sp500['date'] = pd.to_datetime(sp500['Date'], format='%d/%m/%Y', errors='coerce')
sp500['sp500_price'] = pd.to_numeric(sp500['sp500_price'], errors='coerce')
sp500 = sp500[['date', 'sp500_price']].dropna().sort_values('date').reset_index(drop=True)
display(sp500.head(3))

### New Candidate Fundamentals — Data Download

WTI crude oil price (FRED: DCOILWTICO) and the ICE BofA US High Yield Option-Adjusted Spread (FRED: BAMLH0A0HYM2) are downloaded via `pandas_datareader` and saved as local CSVs. The HY spread series starts on December 31, 1996. Subsequent cells load from the saved CSVs rather than re-downloading. Run this cell once to populate the files.

In [ ]:
import os
import pandas_datareader.data as web
import datetime

os.makedirs('potential fundamentals', exist_ok=True)

_start = datetime.date(1985, 1, 1)
_end   = datetime.date(2026, 3, 16)

wti_raw = web.DataReader('DCOILWTICO', 'fred', _start, _end)
wti_raw.index.name = 'Date'
wti_raw.columns = ['wti_price']
wti_raw.to_csv('potential fundamentals/wti_crude.csv')

hy_raw = web.DataReader('BAMLH0A0HYM2', 'fred', _start, _end)
hy_raw.index.name = 'Date'
hy_raw.columns = ['hy_spread']
hy_raw.to_csv('potential fundamentals/high_yield_credit_spread.csv')

display(wti_raw.tail(3))
display(hy_raw.tail(3))

In [ ]:
wti_df = pd.read_csv('potential fundamentals/wti_crude.csv')
wti_df['date'] = pd.to_datetime(wti_df['Date'])
wti_df['wti_price'] = pd.to_numeric(wti_df['wti_price'], errors='coerce')
wti_df['wti'] = np.log(wti_df['wti_price'])
wti_df = wti_df[['date', 'wti']].dropna().sort_values('date').reset_index(drop=True)
display(wti_df.head(3))

In [ ]:
hy_df = pd.read_csv('potential fundamentals/high_yield_credit_spread.csv')
hy_df['date'] = pd.to_datetime(hy_df['Date'])
hy_df['hy_spread'] = pd.to_numeric(hy_df['hy_spread'], errors='coerce')
hy_df = hy_df[['date', 'hy_spread']].dropna().sort_values('date').reset_index(drop=True)
display(hy_df.head(3))

In [ ]:
daily = (
    enrs.set_index('date')
    .join(t1y.set_index('date'), how='outer')
    .join(bpe.set_index('date'), how='outer')
    .join(pe.set_index('date'), how='outer')
    .join(sp500.set_index('date'), how='outer')
    .join(wti_df.set_index('date'), how='outer')
    .join(hy_df.set_index('date'), how='outer')
    .sort_index()
)
display(daily.head())
display(daily.tail())

In [ ]:
audit = pd.DataFrame({
    'first_date': daily.apply(lambda c: c.dropna().index.min().date()),
    'last_date':  daily.apply(lambda c: c.dropna().index.max().date()),
    'n_obs':      daily.count(),
    'n_missing':  daily.isna().sum(),
    'pct_missing': (daily.isna().mean() * 100).round(2)
})
display(audit)

signal_cols = ['enrs', 't1y', 'sp500_best_pe', 'sp500_pe', 'wti', 'hy_spread']
overlap_mask = daily[signal_cols].notna().all(axis=1)
overlap_daily = daily.loc[overlap_mask]
overlap_info = pd.DataFrame({
    'start':            [overlap_daily.index.min().date()],
    'end':              [overlap_daily.index.max().date()],
    'calendar_days':    [(overlap_daily.index.max() - overlap_daily.index.min()).days],
    'business_days':    [len(overlap_daily)]
}, index=['signal overlap'])
display(overlap_info)

The `hy_spread` series begins approximately January 1997, establishing the start of the extended usable overlap window. The extended analysis training period spans approximately 21 years (1997–2018), compared with approximately 28 years in the original analysis. This reduction of roughly 6.5 years results from the HY spread data availability constraint. The training period remains sufficient for reliable estimation; rolling windows of up to 156 weeks are well within the available training sample.

### 4.2 Weekly Aggregation Convention

The analysis is conducted at weekly frequency. Daily data are aggregated to weekly observations using `resample('W-FRI').last()`, which assigns each calendar week to its Friday endpoint and takes the last available non-null observation on or before that Friday.

This convention is defensible for three reasons. First, it uses only data observed at or before end-of-week, so no within-week future information is introduced. Second, if Friday falls on a market holiday, the prior trading day's value is used — consistent with how a practitioner would observe the data. Third, aligning all series to a common Friday date reduces the cross-series asynchrony present in the daily data without introducing any look-ahead.

The weekly dataset is restricted to the overlap window identified above and further filtered to rows where all six signal variables are non-null.

In [ ]:
weekly = daily.resample('W-FRI').last()
weekly = weekly.loc[overlap_daily.index.min():overlap_daily.index.max()]
weekly = weekly.dropna(subset=signal_cols)

display(weekly.shape)
display(weekly.head())
display(weekly.tail())

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)

axes[0].plot(weekly.index, weekly['enrs'], linewidth=0.8, color='steelblue')
axes[0].set_ylabel('S5ENRS Price')
axes[0].set_title('Weekly Data Audit — Signal Series and Benchmark', fontsize=11, loc='left')

axes[1].plot(weekly.index, weekly['t1y'], linewidth=0.8, color='darkorange')
axes[1].set_ylabel('1Y Treasury (%)')

axes[2].plot(weekly.index, weekly['sp500_best_pe'], linewidth=0.8, color='seagreen', label='Best P/E')
axes[2].plot(weekly.index, weekly['sp500_pe'], linewidth=0.8, color='firebrick', linestyle='--', label='Trailing P/E')
axes[2].set_ylabel('P/E Ratio')
axes[2].legend(fontsize=8)

ax4a = axes[3]
ax4b = ax4a.twinx()
ax4a.plot(weekly.index, weekly['wti'], linewidth=0.8, color='saddlebrown', label='log(WTI)')
ax4b.plot(weekly.index, weekly['hy_spread'], linewidth=0.8, color='mediumpurple', linestyle='--', label='HY Spread')
ax4a.set_ylabel('log(WTI)', color='saddlebrown')
ax4b.set_ylabel('HY Spread (%)', color='mediumpurple')
lines1, labels1 = ax4a.get_legend_handles_labels()
lines2, labels2 = ax4b.get_legend_handles_labels()
ax4a.legend(lines1 + lines2, labels1 + labels2, fontsize=8)
ax4a.set_xlabel('Date')

plt.tight_layout()
plt.show()

### 4.3 Forecast Target Construction

The forecast target is the one-week-ahead log return of the S5ENRS Index. At each week $t$, the target is defined as:

$$r_{t+1} = \frac{P_{t+1} - P_t}{P_t}$$

where $P_t$ denotes the S5ENRS weekly close. This is implemented by computing the one-period percentage change and shifting one period forward (`shift(-1)`), so that the value stored in row $t$ is the return that will be realized in period $t+1$. The final row of the dataset will have a NaN target because no subsequent observation exists; this row is not dropped here but must be excluded from any model fitting or evaluation step.

The `enrs_ret` column records the contemporaneous weekly return (realized at $t$). The `target` column records the forward return (to be predicted using information available at $t$).

In [ ]:
weekly = weekly.copy()
weekly['enrs_ret'] = weekly['enrs'].pct_change()
weekly['target'] = weekly['enrs_ret'].shift(-1)

display(weekly[['enrs', 'enrs_ret', 'target']].head(10))
display(weekly[['enrs', 'enrs_ret', 'target']].tail(5))

### 4.4 Train/Test Split

A date-based split is used rather than a row-proportion split. The split date is **January 5, 2018** (first Friday of 2018).

This date was chosen for the following reasons. First, it provides more than eight years in the test set, well above the seven-year minimum. Second, the test window spans several distinct macroeconomic regimes — late-cycle expansion (2018–2019), the COVID shock and rapid recovery (2020–2021), the commodity supercycle and rate normalization episode (2022), and subsequent stabilization — making it a demanding and informative out-of-sample evaluation. The 2022 episode in particular, during which the energy sector experienced extreme price dislocations, is a meaningful stress test for any energy-sector strategy. Third, the training set is long enough to support walk-forward optimization with meaningful rolling windows.

Note that the training period is shorter than in the original analysis (approximately 21 years vs. 28 years) due to the HY spread availability constraint beginning in 1997. Despite this reduction, the training window remains adequate for estimation. No information from the test set is used at any point before this split boundary.

In [ ]:
SPLIT_DATE = pd.Timestamp('2018-01-05')

train = weekly.loc[weekly.index < SPLIT_DATE].copy()
test  = weekly.loc[weekly.index >= SPLIT_DATE].copy()

split_summary = pd.DataFrame({
    'start':   [train.index.min().date(), test.index.min().date()],
    'end':     [train.index.max().date(), test.index.max().date()],
    'n_weeks': [len(train), len(test)],
    'years':   [round(len(train) / 52, 1), round(len(test) / 52, 1)]
}, index=['train', 'test'])
display(split_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(train.index, train['enrs'], color='steelblue', linewidth=0.8, label='Train')
ax.plot(test.index,  test['enrs'],  color='firebrick', linewidth=0.8, label='Test')
ax.axvline(SPLIT_DATE, color='black', linestyle='--', linewidth=1)
ax.set_ylabel('S5ENRS Price')
ax.set_title('S5ENRS — Train / Test Split', fontsize=11, loc='left')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 4.5 Strategy Design and Economic Logic

Both strategies attempt to exploit mean-reversion in the gap between the observed S5ENRS price and a model-implied fair value. The economic rationale is that energy sector equity prices contain a component driven by broad market valuation conditions, the interest rate environment, commodity prices, and credit conditions, and that sustained deviations from model-implied fair value tend to partially reverse over subsequent periods.

**Strategy 1 — Rolling Regression Fair Value Gap.** At each week $t$, a rolling OLS regression of $\log(\text{S5ENRS}_t)$ on the selected inputs is estimated using the preceding $W$ weeks. The fitted value constitutes the model's log fair value estimate. The fair value gap is the signed difference between observed log price and fitted log price. When this gap is sufficiently negative (price below fair value, as measured by an expanding-window z-score threshold), a long position is entered for the following week; when the gap is sufficiently positive, a short position is entered; otherwise the strategy is flat.

**Strategy 2 — Kalman Filter Fair Value Gap.** The same log-price regression is re-estimated as a state-space system in which the OLS coefficients are replaced by a time-varying state vector that evolves as a random walk. The Kalman filter recursion updates the coefficient estimates at each $t$ using only data available through $t-1$, and the fair value estimate is the one-step-ahead prediction. This is leakage-safe by construction.

**Tuning rule.** Hyperparameters are selected using cumulative return over the **training sample only**. Feature set selection is also conducted on training data only, using the same cumulative return objective over a fixed window of 104 weeks and threshold of 1.0. Six candidate feature specifications are evaluated, including the original baseline and four extensions that add one or both new candidate fundamentals. No test-period data enters any selection or tuning step.

### 4.6 Strategy 1: Rolling Regression Fair Value Gap

#### Variable Selection

Six candidate feature specifications are evaluated on the training sample. The six specifications test: the original baseline (1-year Treasury yield plus trailing P/E), the baseline with forward P/E added, and four combinations incorporating WTI log price, HY spread, or both. Using a fixed window of 104 weeks and a threshold of 1.0, each specification's training cumulative return is computed. The specification with the highest training cumulative return is carried forward to the window and threshold grid search. If the baseline specification is selected, this indicates the new candidate fundamentals did not improve training performance under the fixed evaluation criteria.

In [ ]:
def rolling_reg_signals(df, features, window, threshold, min_gap_periods=52):
    log_price = np.log(df['enrs'].values)
    Xmat      = df[features].values
    n         = len(df)
    fitted    = np.full(n, np.nan)
    for t in range(window, n):
        y_w  = log_price[t - window:t]
        X_w  = np.column_stack([np.ones(window), Xmat[t - window:t]])
        beta = np.linalg.lstsq(X_w, y_w, rcond=None)[0]
        fitted[t] = np.r_[1.0, Xmat[t]] @ beta
    gap_s  = pd.Series(log_price - fitted, index=df.index)
    g_mean = gap_s.expanding(min_periods=min_gap_periods).mean()
    g_std  = gap_s.expanding(min_periods=min_gap_periods).std().replace(0, np.nan)
    z_gap  = (gap_s - g_mean) / g_std
    sig    = np.where(z_gap.isna(),       0.0,
             np.where(z_gap < -threshold, 1.0,
             np.where(z_gap >  threshold, -1.0, 0.0)))
    return pd.Series(sig, index=df.index), gap_s, pd.Series(fitted, index=df.index)

def strat_cum_ret(signals, target):
    return ((1 + (signals * target).fillna(0)).prod() - 1) * 100

feature_specs = {
    'baseline (t1y + pe)':           ['t1y', 'sp500_pe'],
    'baseline + best_pe':            ['t1y', 'sp500_best_pe', 'sp500_pe'],
    'baseline + wti':                ['t1y', 'sp500_pe', 'wti'],
    'baseline + hy_spread':          ['t1y', 'sp500_pe', 'hy_spread'],
    'baseline + wti + hy_spread':    ['t1y', 'sp500_pe', 'wti', 'hy_spread'],
    'all fundamentals':              ['t1y', 'sp500_best_pe', 'sp500_pe', 'wti', 'hy_spread'],
}

varsel_rows = [
    {'spec': name,
     'train_cum_ret (%)': round(
         strat_cum_ret(rolling_reg_signals(train, feats, 104, 1.0)[0],
                       train['target']), 2)}
    for name, feats in feature_specs.items()
]
varsel_df = (pd.DataFrame(varsel_rows)
               .sort_values('train_cum_ret (%)', ascending=False)
               .reset_index(drop=True))
display(varsel_df)

ROLL_FEATURES = feature_specs[varsel_df.iloc[0]['spec']]

In [ ]:
roll_windows_dense    = list(np.round(np.linspace(26, 260, 50)).astype(int))
roll_thresholds_dense = list(np.round(np.linspace(0.25, 2.5, 50), 4))

roll_grid = np.full((len(roll_thresholds_dense), len(roll_windows_dense)), np.nan)
for i, thr in enumerate(roll_thresholds_dense):
    for j, win in enumerate(roll_windows_dense):
        sigs, _, _ = rolling_reg_signals(train, ROLL_FEATURES, win, thr)
        roll_grid[i, j] = strat_cum_ret(sigs, train['target'])

_roll_smooth = np.empty_like(roll_grid)
for _i in range(roll_grid.shape[0]):
    for _j in range(roll_grid.shape[1]):
        _roll_smooth[_i, _j] = np.nanmean(
            roll_grid[max(0, _i-1):_i+2, max(0, _j-1):_j+2])

bi, bj         = np.unravel_index(np.nanargmax(_roll_smooth), _roll_smooth.shape)
ROLL_WINDOW    = roll_windows_dense[bj]
ROLL_THRESHOLD = round(roll_thresholds_dense[bi], 4)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(roll_grid, aspect='auto', cmap='RdYlGn', origin='lower')
_rx = list(range(0, len(roll_windows_dense), 5))
_ry = list(range(0, len(roll_thresholds_dense), 5))
ax.set_xticks(_rx)
ax.set_xticklabels([roll_windows_dense[k] for k in _rx], fontsize=7)
ax.set_yticks(_ry)
ax.set_yticklabels([f'{roll_thresholds_dense[k]:.2f}' for k in _ry], fontsize=7)
ax.set_xlabel('Rolling window (weeks)')
ax.set_ylabel('Signal threshold')
ax.set_title('Rolling Regression — 50×50 Training Cumulative Return (%) — Smoothed peak selected',
             fontsize=11, loc='left')
ax.scatter([bj], [bi], marker='*', color='black', s=120, zorder=5,
           label=f'Selected: window={ROLL_WINDOW}, threshold={ROLL_THRESHOLD}')
ax.legend(fontsize=8)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

#### Selected Specification

The window and threshold are selected from the 50 × 50 dense grid using the 3 × 3 neighbourhood-smoothed surface, which prefers the most consistently high-performing region over an isolated best cell. `ROLL_FEATURES`, `ROLL_WINDOW`, and `ROLL_THRESHOLD` are set automatically from that selection. The cell below displays the chosen parameters, generates signals over the full weekly sample, and produces a compact audit table.

In [ ]:
spec_summary = pd.DataFrame({
    'features':  [', '.join(ROLL_FEATURES)],
    'window':    [ROLL_WINDOW],
    'threshold': [ROLL_THRESHOLD],
}, index=['rolling regression'])
display(spec_summary)

roll_sigs, roll_gap, roll_fv = rolling_reg_signals(
    weekly, ROLL_FEATURES, ROLL_WINDOW, ROLL_THRESHOLD)

audit_roll = pd.DataFrame({
    'enrs':       weekly['enrs'],
    'fair_value': np.exp(roll_fv).round(3),
    'gap':        roll_gap.round(4),
    'signal':     roll_sigs,
    'next_ret':   weekly['target'].round(4),
}).dropna(subset=['fair_value'])
display(audit_roll.head(10))
display(audit_roll.tail(5))

### 4.7 Strategy 2: Kalman Filter Fair Value Gap

The Kalman filter state-space model has two classes of hyperparameters that require explicit optimisation:

- **Internal Kalman hyperparameters** — the process noise scale $q$ and the observation noise $r$. These govern the filter dynamics. $q$ controls how quickly the coefficient vector is permitted to change; $r$ controls the measurement uncertainty. Together they determine the Kalman gain and the speed of coefficient adaptation.

- **External trading rule hyperparameter** — the z-score threshold that converts the fair value gap into long, flat, and short positions.

All three parameters are optimised jointly on the training sample using a two-stage search. Stage 1 scans a broad 15 × 15 × 15 log-scale grid (3,375 combinations) to locate the high-performing region of the $(q,\,r,\,\text{threshold})$ space. Stage 2 performs a dense 20 × 20 × 20 local search (8,000 combinations) centred on the Stage 1 optimum, with the final $(q, r)$ pair taken from the 3 × 3 neighbourhood-smoothed Stage 2 surface. The feature set is taken from the rolling regression variable selection above (`KF_FEATURES = ROLL_FEATURES`) for direct comparability. No test-period information enters the search at any stage.

In [ ]:
def kalman_tvr_signals(df, features, q, r, threshold, min_gap_periods=52):
    log_price = np.log(df['enrs'].values)
    Xmat      = np.column_stack([np.ones(len(df))] + [df[f].values for f in features])
    n, p      = Xmat.shape
    beta      = np.zeros(p)
    P         = np.eye(p) * 10.0
    Q         = np.eye(p) * q
    fitted    = np.full(n, np.nan)
    for t in range(n):
        x_t    = Xmat[t]
        P_pred = P + Q
        y_hat  = float(x_t @ beta)
        S      = float(x_t @ P_pred @ x_t) + r
        K      = (P_pred @ x_t) / S
        beta   = beta + K * (log_price[t] - y_hat)
        P      = (np.eye(p) - np.outer(K, x_t)) @ P_pred
        fitted[t] = y_hat
    gap_s  = pd.Series(log_price - fitted, index=df.index)
    g_mean = gap_s.expanding(min_periods=min_gap_periods).mean()
    g_std  = gap_s.expanding(min_periods=min_gap_periods).std().replace(0, np.nan)
    z_gap  = (gap_s - g_mean) / g_std
    sig    = np.where(z_gap.isna(),       0.0,
             np.where(z_gap < -threshold, 1.0,
             np.where(z_gap >  threshold, -1.0, 0.0)))
    return pd.Series(sig, index=df.index), gap_s, pd.Series(fitted, index=df.index)

KF_FEATURES = ROLL_FEATURES

kf_q_s1 = list(np.logspace(-8, -1, 15))
kf_r_s1 = list(np.logspace(-2, 1.3, 15))
kf_t_s1 = list(np.round(np.linspace(0.25, 2.5, 15), 3))

kf_grid_s1 = np.full((len(kf_t_s1), len(kf_q_s1), len(kf_r_s1)), np.nan)
for i, thr in enumerate(kf_t_s1):
    for j, q_val in enumerate(kf_q_s1):
        for m, r_val in enumerate(kf_r_s1):
            sigs, _, _ = kalman_tvr_signals(train, KF_FEATURES, q_val, r_val, thr)
            kf_grid_s1[i, j, m] = strat_cum_ret(sigs, train['target'])

bi_s1, bj_s1, bm_s1 = np.unravel_index(np.nanargmax(kf_grid_s1), kf_grid_s1.shape)
_kf_q_c = kf_q_s1[bj_s1]
_kf_r_c = kf_r_s1[bm_s1]
_kf_t_c = kf_t_s1[bi_s1]

kf_q_s2 = list(np.logspace(np.log10(_kf_q_c) - 1.0, np.log10(_kf_q_c) + 1.0, 20))
kf_r_s2 = list(np.logspace(np.log10(max(1e-3, _kf_r_c / 6)),
                            np.log10(min(50.0,  _kf_r_c * 6)), 20))
kf_t_s2 = list(np.round(np.linspace(max(0.1, _kf_t_c - 0.6),
                                     min(3.0, _kf_t_c + 0.6), 20), 4))

kf_grid_s2 = np.full((len(kf_t_s2), len(kf_q_s2), len(kf_r_s2)), np.nan)
for i, thr in enumerate(kf_t_s2):
    for j, q_val in enumerate(kf_q_s2):
        for m, r_val in enumerate(kf_r_s2):
            sigs, _, _ = kalman_tvr_signals(train, KF_FEATURES, q_val, r_val, thr)
            kf_grid_s2[i, j, m] = strat_cum_ret(sigs, train['target'])

kf_qr_s2 = kf_grid_s2.max(axis=0)

_kf_qr_smooth = np.empty_like(kf_qr_s2)
for _i in range(kf_qr_s2.shape[0]):
    for _j in range(kf_qr_s2.shape[1]):
        _kf_qr_smooth[_i, _j] = np.nanmean(
            kf_qr_s2[max(0, _i-1):_i+2, max(0, _j-1):_j+2])

bj_s2, bm_s2 = np.unravel_index(np.nanargmax(_kf_qr_smooth), _kf_qr_smooth.shape)
KF_Q         = kf_q_s2[bj_s2]
KF_R         = kf_r_s2[bm_s2]
KF_THRESHOLD = kf_t_s2[int(np.nanargmax(kf_grid_s2[:, bj_s2, bm_s2]))]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(kf_qr_s2, aspect='auto', cmap='RdYlGn', origin='lower')
_kq = list(range(0, len(kf_q_s2), 4))
_kr = list(range(0, len(kf_r_s2), 4))
ax.set_yticks(_kq)
ax.set_yticklabels([f'{kf_q_s2[k]:.1e}' for k in _kq], fontsize=7)
ax.set_xticks(_kr)
ax.set_xticklabels([f'{kf_r_s2[k]:.3f}' for k in _kr], fontsize=7, rotation=30)
ax.set_xlabel('Observation noise R  (Stage 2 refined range)')
ax.set_ylabel('Process noise scale Q  (Stage 2 refined range)')
ax.set_title('Kalman Filter — Stage 2 Best Training Cum. Return (%) over threshold — Smoothed peak selected',
             fontsize=10, loc='left')
ax.scatter([bm_s2], [bj_s2], marker='*', color='black', s=120, zorder=5,
           label=f'Q={KF_Q:.2e},  R={KF_R:.3f},  thr={KF_THRESHOLD}')
ax.legend(fontsize=8)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

#### Selected Specification

Stage 1 (broad 15 × 15 × 15 log-scale scan) identifies the high-performing region of the $(Q, R, \\text{threshold})$ space. Stage 2 (dense 20 × 20 × 20 local search centred on the Stage 1 optimum) refines within that region. The heatmap shows the Stage 2 $(Q, R)$ surface with the best training cumulative return maximised over the refined threshold range. `KF_Q`, `KF_R`, and `KF_THRESHOLD` are selected from the 3 × 3 neighbourhood-smoothed Stage 2 surface, with the threshold set to the best value at the chosen $(Q, R)$ point. The cell below applies the filter to the full weekly sample using the selected specification and produces an audit table.

In [ ]:
kf_spec_summary = pd.DataFrame({
    'features':  [', '.join(KF_FEATURES)],
    'Q scale':   [KF_Q],
    'R':         [KF_R],
    'threshold': [KF_THRESHOLD],
}, index=['kalman filter'])
display(kf_spec_summary)

kf_sigs, kf_gap, kf_fv = kalman_tvr_signals(
    weekly, KF_FEATURES, KF_Q, KF_R, KF_THRESHOLD)

audit_kf = pd.DataFrame({
    'enrs':       weekly['enrs'],
    'fair_value': np.exp(kf_fv).round(3),
    'gap':        kf_gap.round(4),
    'signal':     kf_sigs,
    'next_ret':   weekly['target'].round(4),
})
display(audit_kf.head(10))
display(audit_kf.tail(5))

### 4.8 Walk-Forward Reoptimization

Both strategy functions implement continuous sequential estimation that is walk-forward safe by construction. The rolling regression refits OLS coefficients each week using only the trailing $W$ weeks of data, and the Kalman filter updates its state vector each period using only data observed through $t-1$. Both models therefore satisfy the minimum annual reoptimization requirement, as the underlying estimates are refreshed at every weekly step throughout the test window.

**Hyperparameter stability.** The rolling window length, process noise scale, observation noise, and signal threshold selected on the training sample are held fixed throughout the test period. Re-selecting hyperparameters at each walk-forward origin would require an additional inner grid search and would introduce complexity without clear benefit, given that the training regime already spans 21+ years. Holding parameters fixed is the simpler and more defensible choice.

**Signal extraction for the test period.** Applying `rolling_reg_signals` and `kalman_tvr_signals` to the full weekly dataset produces walk-forward valid signals for the entire test period: at any test week $t$, the rolling regression uses only the trailing $W$ observations and the Kalman filter state is conditioned only on data through $t-1$. Test-period signals are extracted directly from these pre-computed series. No test-period data entered the model at any earlier stage.

In [ ]:
weekly['sp500_ret']    = weekly['sp500_price'].pct_change()
weekly['sp500_target'] = weekly['sp500_ret'].shift(-1)

test_idx   = weekly.index >= SPLIT_DATE
test_dates = weekly.index[test_idx]

ret_roll  = (roll_sigs.loc[test_dates] * weekly.loc[test_dates, 'target']).fillna(0)
ret_kf    = (kf_sigs.loc[test_dates]   * weekly.loc[test_dates, 'target']).fillna(0)
ret_bh    = weekly.loc[test_dates, 'target'].fillna(0)
ret_sp500 = weekly.loc[test_dates, 'sp500_target'].fillna(0)

ret_series = {
    'Rolling Regression (Extended)': ret_roll,
    'Kalman Filter (Extended)':      ret_kf,
    'BnH S5ENRS':                    ret_bh,
    'S&P 500':                       ret_sp500,
}
PLOT_COLORS = ['steelblue', 'firebrick', 'darkorange', 'seagreen']

### 4.9 Performance Evaluation

Strategy returns are computed over the test period only. The risk-free rate is the average 1-year Treasury yield over the test period, converted to a weekly equivalent ($r_f / 52$). All four series — the two strategies, the S5ENRS buy-and-hold, and the S&P 500 benchmark — use identical weekly return timing: the position or index exposure held from Friday close at $t$ earns the return to Friday close at $t+1$.

The **Gini coefficient** is computed on the distribution of absolute weekly returns. A value near zero indicates returns are spread roughly evenly across weeks; a value near one indicates that returns are concentrated in a small number of periods. Concentration implies return fragility — the strategy's aggregate performance depends on a limited set of weeks rather than consistent positive contributions.

The **hit rate** for the two trading strategies measures directional accuracy over active signal weeks only (weeks where the signal is non-zero). For the buy-and-hold benchmarks, the reported value corresponds to the share of weeks with positive next-period returns.

In [ ]:
rf_annual = weekly.loc[test_dates, 't1y'].mean() / 100
rf_weekly = rf_annual / 52

def ann_return(r, freq=52):
    r = r.dropna()
    if len(r) == 0: return np.nan
    return (1 + r).prod() ** (freq / len(r)) - 1

def ann_vol(r, freq=52):
    return r.dropna().std() * np.sqrt(freq)

def sharpe(r, rf, freq=52):
    ex = r.dropna() - rf
    return ex.mean() / ex.std() * np.sqrt(freq) if ex.std() != 0 else np.nan

def sortino(r, rf, freq=52):
    ex  = r.dropna() - rf
    ds  = ex[ex < 0].std()
    return ex.mean() / ds * np.sqrt(freq) if ds != 0 else np.nan

def max_dd(r):
    w = (1 + r.fillna(0)).cumprod()
    return ((w - w.cummax()) / w.cummax()).min()

def hit_rate(sig, tgt):
    mask = (sig != 0) & tgt.notna()
    if mask.sum() == 0: return np.nan
    return ((sig[mask] * tgt[mask]) > 0).sum() / mask.sum()

def gini(r):
    a = np.sort(np.abs(r.dropna()))
    n = len(a)
    if n == 0 or a.sum() == 0: return np.nan
    return (2 * (np.arange(1, n + 1) * a).sum()) / (n * a.sum()) - (n + 1) / n

tgt     = weekly.loc[test_dates, 'target']
sig_map = {
    'Rolling Regression (Extended)': roll_sigs.loc[test_dates],
    'Kalman Filter (Extended)':      kf_sigs.loc[test_dates],
    'BnH S5ENRS':                    pd.Series(1.0, index=test_dates),
    'S&P 500':                       pd.Series(1.0, index=test_dates),
}

perf = pd.DataFrame({
    name: {
        'Ann. Return (%)':  round(ann_return(r) * 100, 2),
        'Ann. Vol (%)':     round(ann_vol(r) * 100, 2),
        'Sharpe':           round(sharpe(r, rf_weekly), 3),
        'Sortino':          round(sortino(r, rf_weekly), 3),
        'Max Drawdown (%)': round(max_dd(r) * 100, 2),
        'Hit Rate (%)':     round(hit_rate(sig_map[name], tgt) * 100, 2),
        'Gini':             round(gini(r), 3),
    }
    for name, r in ret_series.items()
}).T

display(perf)

---

## 5. Results

### 5.1 Strategy Diagnostic Plots — Training Period

The following plots show the model fair value estimate alongside the realized S5ENRS price, the fair value gap, and the resulting signal series over the training period. These are a sanity check on model behavior prior to out-of-sample evaluation.

In [ ]:
tr = weekly.index < SPLIT_DATE

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(weekly.index[tr], weekly['enrs'].values[tr],
             color='steelblue', linewidth=0.8, label='S5ENRS')
axes[0].plot(weekly.index[tr], np.exp(roll_fv.values[tr]),
             color='darkorange', linewidth=0.8, linestyle='--', label='Fair value')
axes[0].set_ylabel('Price')
axes[0].set_title('Strategy 1: Rolling Regression (Extended) — Training Period', fontsize=11, loc='left')
axes[0].legend(fontsize=8)

axes[1].plot(weekly.index[tr], roll_gap.values[tr], color='purple', linewidth=0.7)
axes[1].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[1].set_ylabel('Log gap')

axes[2].step(weekly.index[tr], roll_sigs.values[tr],
             color='firebrick', linewidth=0.7, where='post')
axes[2].set_yticks([-1, 0, 1])
axes[2].set_yticklabels(['Short', 'Flat', 'Long'])
axes[2].set_ylabel('Signal')
axes[2].set_xlabel('Date')

plt.tight_layout()
plt.show()

In [ ]:
tr = weekly.index < SPLIT_DATE

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(weekly.index[tr], weekly['enrs'].values[tr],
             color='steelblue', linewidth=0.8, label='S5ENRS')
axes[0].plot(weekly.index[tr], np.exp(kf_fv.values[tr]),
             color='darkorange', linewidth=0.8, linestyle='--', label='Fair value')
axes[0].set_ylabel('Price')
axes[0].set_title('Strategy 2: Kalman Filter (Extended) — Training Period', fontsize=11, loc='left')
axes[0].legend(fontsize=8)

axes[1].plot(weekly.index[tr], kf_gap.values[tr], color='purple', linewidth=0.7)
axes[1].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[1].set_ylabel('Log gap')

axes[2].step(weekly.index[tr], kf_sigs.values[tr],
             color='firebrick', linewidth=0.7, where='post')
axes[2].set_yticks([-1, 0, 1])
axes[2].set_yticklabels(['Short', 'Flat', 'Long'])
axes[2].set_ylabel('Signal')
axes[2].set_xlabel('Date')

plt.tight_layout()
plt.show()

### 5.2 Out-of-Sample Test Period Results

The following plots and table summarise strategy performance over the test period (January 2018 — March 2026). All returns are out-of-sample. The position at week $t$ uses only information available through $t$, and the resulting return is realised in week $t+1$. The S&P 500 and S5ENRS buy-and-hold series use identical return timing for a fair comparison.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for (name, r), col in zip(ret_series.items(), PLOT_COLORS):
    cum = (1 + r.fillna(0)).cumprod()
    ax.plot(cum.index, cum.values, linewidth=0.9, color=col, label=name)

ax.axhline(1, color='black', linewidth=0.4, linestyle='--')
ax.set_ylabel('Growth of $1')
ax.set_title('Cumulative Returns — Out-of-Sample Test Period', fontsize=11, loc='left')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

for (name, r), col in zip(ret_series.items(), PLOT_COLORS):
    w  = (1 + r.fillna(0)).cumprod()
    dd = (w - w.cummax()) / w.cummax()
    ax.plot(dd.index, dd.values * 100, linewidth=0.8, color=col, label=name)

ax.axhline(0, color='black', linewidth=0.4)
ax.set_ylabel('Drawdown (%)')
ax.set_title('Drawdown — Out-of-Sample Test Period', fontsize=11, loc='left')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

axes[0].step(test_dates, roll_sigs.loc[test_dates].values,
             color='steelblue', linewidth=0.7, where='post')
axes[0].set_yticks([-1, 0, 1])
axes[0].set_yticklabels(['Short', 'Flat', 'Long'])
axes[0].set_title('Rolling Regression (Extended) — Signal (Test Period)', fontsize=10, loc='left')

axes[1].step(test_dates, kf_sigs.loc[test_dates].values,
             color='firebrick', linewidth=0.7, where='post')
axes[1].set_yticks([-1, 0, 1])
axes[1].set_yticklabels(['Short', 'Flat', 'Long'])
axes[1].set_title('Kalman Filter (Extended) — Signal (Test Period)', fontsize=10, loc='left')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

In [ ]:
rng = np.random.default_rng(42)
tgt_vals = weekly.loc[test_dates, 'target'].values

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, (name, sigs, col) in zip(axes, [
    ('Rolling Regression (Extended)', roll_sigs.loc[test_dates].values, 'steelblue'),
    ('Kalman Filter (Extended)',      kf_sigs.loc[test_dates].values,   'firebrick'),
]):
    mask = ~np.isnan(tgt_vals)
    jitter = rng.uniform(-0.12, 0.12, mask.sum())
    ax.scatter(sigs[mask] + jitter, tgt_vals[mask],
               alpha=0.25, s=5, color=col, linewidths=0)
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_xticks([-1, 0, 1])
    ax.set_xticklabels(['Short', 'Flat', 'Long'])
    ax.set_xlabel('Signal at $t$')
    ax.set_ylabel('Return at $t+1$')
    ax.set_title(name, fontsize=10, loc='left')

fig.suptitle('Signal vs. Next-Period Return — Test Period', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

The scatter plots above show the distribution of next-period returns conditional on the signal at time $t$. Substantial overlap across signal categories is expected given the low signal-to-noise ratio typical of weekly equity return forecasting.

The performance table and plots summarise out-of-sample results over the test period (January 2018 — March 2026).

**Feature selection outcome.** The variable selection step evaluated six candidate specifications on the training sample using cumulative return at a fixed window of 104 weeks and threshold of 1.0. The specification selected is reported in the table above. If the selected set includes `wti` or `hy_spread`, those variables demonstrated incremental training-period value; if the baseline was returned, the new variables did not improve upon the original inputs under this criterion.

**Hyperparameter search.** Rolling regression used a 50 × 50 dense grid (2,500 combinations) with 3 × 3 neighbourhood-smoothed peak selection. The Kalman filter used a two-stage search (15 × 15 × 15 broad scan followed by a 20 × 20 × 20 local refinement) with neighbourhood-smoothed Stage 2 selection. All tuning is training-sample only.

Both strategies generate directional signals rather than continuously investing, so their return profiles differ structurally from buy-and-hold. Relative to the S5ENRS buy-and-hold, any reduction in annualised volatility or maximum drawdown comes at the cost of reduced market exposure during recoveries. The 2020 COVID crash and the 2022 energy sector repricing are the dominant episodes shaping test-period performance. The Gini coefficient indicates the degree to which each strategy’s total return is concentrated in a small number of weeks. Hit rates above 50 % over active signal weeks are consistent with some directional forecasting contribution, though statistical significance is not assessed here.

### Limitations

- **No transaction costs.** All returns are gross of trading costs. Weekly rebalancing and short exposure in an equity index product carry bid-ask spreads and potential borrowing costs not modelled here.
- **Threshold sensitivity.** Signal thresholds and hyperparameters were selected by maximising training cumulative return. The selected values may be locally optimal for the training regime and could underperform in regimes not well-represented in that window.
- **Fundamental inputs.** The model uses a limited set of macroeconomic inputs. Energy sector prices are also materially driven by geopolitical events and sector-specific supply dynamics not captured here.
- **Fixed parameters across the test period.** Neither strategy re-selects hyperparameters at walk-forward origins during the test window.
- **Short selling frictions.** The strategy assumes S5ENRS can be shorted without friction, which may not reflect actual shorting costs or index replication constraints.
- **Extended overlap window.** Incorporating `hy_spread` restricts the overlap to post-1997, reducing the training period by approximately 6.5 years relative to the original analysis. Results may differ from the original notebook due to this shorter training window, in addition to any effect of the expanded candidate feature set.

In [ ]:
import os
os.makedirs('figures_extended', exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 5))
for (name, r), col in zip(ret_series.items(), PLOT_COLORS):
    cum = (1 + r.fillna(0)).cumprod()
    ax.plot(cum.index, cum.values, linewidth=0.9, color=col, label=name)
ax.axhline(1, color='black', linewidth=0.4, linestyle='--')
ax.set_ylabel('Growth of $1')
ax.set_title('Cumulative Returns — Out-of-Sample Test Period', fontsize=11, loc='left')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures_extended/cumulative_returns.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
for (name, r), col in zip(ret_series.items(), PLOT_COLORS):
    w = (1 + r.fillna(0)).cumprod()
    dd = (w - w.cummax()) / w.cummax()
    ax.plot(dd.index, dd.values * 100, linewidth=0.8, color=col, label=name)
ax.axhline(0, color='black', linewidth=0.4)
ax.set_ylabel('Drawdown (%)')
ax.set_title('Drawdown — Out-of-Sample Test Period', fontsize=11, loc='left')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures_extended/drawdown.png', dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].step(test_dates, roll_sigs.loc[test_dates].values,
             color='steelblue', linewidth=0.7, where='post')
axes[0].set_yticks([-1, 0, 1])
axes[0].set_yticklabels(['Short', 'Flat', 'Long'])
axes[0].set_title('Rolling Regression (Extended) — Signal (Test Period)', fontsize=10, loc='left')
axes[1].step(test_dates, kf_sigs.loc[test_dates].values,
             color='firebrick', linewidth=0.7, where='post')
axes[1].set_yticks([-1, 0, 1])
axes[1].set_yticklabels(['Short', 'Flat', 'Long'])
axes[1].set_title('Kalman Filter (Extended) — Signal (Test Period)', fontsize=10, loc='left')
axes[1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('figures_extended/signals_test.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 6. Conclusions

This notebook extended the original S5ENRS fair value gap analysis by evaluating WTI crude oil log price and the ICE BofA US High Yield Option-Adjusted Spread as additional candidate fundamentals. Both variables were assessed through training-period variable selection alongside the original baseline inputs, using a 50 × 50 dense rolling grid and a two-stage Kalman search to ensure a thorough exploration of the training-sample performance surface.

The variable selection procedure identified the best-performing specification from six candidates using training cumulative return at a fixed window and threshold. The selected specification is reported in the summary table in Section 4.6. Whether the new variables entered the selected set is documented there; that result, rather than any prior expectation, is the operative finding.

The primary structural consequence of including `hy_spread` is a reduction in training data: the overlap window starts in early 1997 rather than the early 1990s, removing approximately 6.5 years from the training sample. This shorter training period affects selected hyperparameter values and training-period performance relative to the original analysis and should be considered when comparing results across notebooks.

The broader conclusion from both the original and extended analyses is that fundamentals-based fair value gap strategies for S5ENRS face meaningful difficulty in the chosen test period. The dominant episodes — the COVID crash and recovery (2020) and the commodity-driven energy rally (2022) — produced large, rapid directional moves that are difficult to anticipate from macro-financial inputs alone. Incorporating commodity prices directly via WTI log price is a conceptually motivated extension; the training-period variable selection results reported above are the appropriate empirical criterion for assessing whether that extension is productive.

---

## What Remains for the Final Written Report

- Compare extended results against the original notebook numerically; attribute differences to (a) the shorter post-1997 training window and (b) any change in selected features
- Write polished prose for the Introduction, Methodology, Results, and Conclusions sections aligned with the project submission guidelines
- Confirm every required metric, visual, and test is present and correctly referenced
- Verify all numerical results are consistent between the notebook and the written report

---

*Hyperparameter search density was updated to match the improved approach used in `final_project.ipynb`: rolling regression uses a 50 × 50 grid with 3 × 3 neighbourhood-smoothed peak selection; the Kalman filter uses a two-stage search (15 × 15 × 15 broad scan followed by a 20 × 20 × 20 dense local refinement). All tuning remains training-sample only. Downstream results and heatmaps were refreshed accordingly.*